# Lab 2: Memory-based CF

## Section 1: Reload / Revisit Movielens 10M dataset and TopPop recommender

#### Dataset: https://grouplens.org/datasets/movielens/10m/

In [1]:
from urllib.request import urlretrieve
import zipfile, os

In [2]:
# If file exists, skip the download
data_file_path = "movielens/Movielens10M/"
data_file_name = data_file_path + "movielens_10m.zip"

# If directory does not exist, create
if not os.path.exists(data_file_path):
    os.makedirs(data_file_path)

if not os.path.exists(data_file_name):
    urlretrieve("http://files.grouplens.org/datasets/movielens/ml-10m.zip", data_file_name)

In [3]:
# unzip and read the ratings.dat file
dataFile = zipfile.ZipFile(data_file_path + "movielens_10m.zip")

mv_path = dataFile.extract("ml-10M100K/ratings.dat", path = data_file_path)

#### Let's take a look at the data

In [4]:
import pandas as pd

In [24]:
mv_df = pd.read_csv(filepath_or_buffer=mv_path, sep="::", header=None, dtype={0:int, 1:int, 2:float, 3:int}, engine='python')

mv_df.columns = ["UserID", "ItemID", "Interaction", "Timestamp"]

In [ ]:
mv_df.head(n=10)

In [ ]:
print("The number of interactions is {}".format(len(mv_df)))

#### We will now create a sparse matrix

<u>Note</u> : we read UserID and ItemID as int, but this is not always the case if the IDs are alphanumeric

#### Let's extract the list of unique user id and item id, and show some statistics to understand our data

In [27]:
userID_unique = mv_df["UserID"].unique()
itemID_unique = mv_df["ItemID"].unique()

In [ ]:
n_users = len(userID_unique)
n_items = len(itemID_unique)
n_interactions = len(mv_df)

print("Number of users: {}\t, Number of items: {}".format(n_users, n_items))
print("Max UserID: {}\t, Max ItemID: {}".format(max(userID_unique), max(itemID_unique)))

#### We observe that the max ID of users and items is higher than the number of unique users and items. Thus, we have some empty profiles

#### We should remove those empty indices, by creating a new mapping

In [29]:
mapped_id, original_id = pd.factorize(mv_df["UserID"].unique())
user_original_ID_to_index = pd.Series(mapped_id, index=original_id)

mapped_id, original_id = pd.factorize(mv_df["ItemID"].unique())
item_original_ID_to_index = pd.Series(mapped_id, index=original_id)

In [ ]:
original_item_ID = 292
print("New index for item {} is {}".format(original_item_ID, item_original_ID_to_index[original_item_ID]))

#### Replacing the IDs in the dataframe and we are ready to use the data

In [31]:
mv_df["UserID"] = mv_df["UserID"].map(user_original_ID_to_index)
mv_df["ItemID"] = mv_df["ItemID"].map(item_original_ID_to_index)

In [ ]:
mv_df.head(n=10)

In [ ]:
userID_unique = mv_df["UserID"].unique()
itemID_unique = mv_df["ItemID"].unique()

n_users = len(userID_unique)
n_items = len(itemID_unique)
n_interactions = len(mv_df)

print("Number of users: {}\t, Number of items: {}".format(n_users, n_items))
print("Max UserID: {}\t, Max ItemID: {}\n".format(max(userID_unique), max(itemID_unique)))

print("Average interactions per user: {:.2f}".format(n_interactions/n_users))
print("Average interactions per item: {:.2f}\n".format(n_interactions/n_items))

print("Sparsity {:.2f} %".format((1-float(n_interactions)/(n_users*n_items))*100))

In [ ]:
import scipy.sparse as sps
import numpy as np

mv_all = sps.coo_matrix((mv_df['Interaction'].values, (mv_df['UserID'].values, mv_df['ItemID'].values)))

mv_all

In [ ]:
item_popularity = np.ediff1d(mv_all.tocsc().indptr)
item_popularity

In [ ]:
ten_percent = int(n_items/10)

print("Average per-item interactions over the whole dataset: {:.2f}".
      format(item_popularity.mean()))

print("Average per-item interactions for the top 10% popular items: {:.2f}".
      format(item_popularity[-ten_percent:].mean()))

print("Average per-item interactions for the least 10% popular items: {:.2f}".
      format(item_popularity[:ten_percent].mean()))

print("Average per-item interactions for the median 10% popular items: {:.2f}".
      format(item_popularity[int(n_items*0.45):int(n_items*0.55)].mean()))

In [ ]:
print("Number of items with zero interactions {}".format(np.sum(item_popularity==0)))

#### Evaluation:

#### In order to evaluate our recommender, we have to:
* Split the data into mv_train and mv_test (i.e., train and test set)
* Define evaluation metrics
* Define a functon to compute the evaluation for each user

<u>Note</u>: The splitting of the data is very important to ensure your algorithm is evaluated appropriately

In [ ]:
train_test_split = 0.80

n_interactions = mv_all.nnz


train_mask = np.random.choice([True, False], n_interactions, p=[train_test_split, 1-train_test_split])
train_mask

In [ ]:
mv_train = sps.csr_matrix((mv_all.data[train_mask], (mv_all.row[train_mask], mv_all.col[train_mask])))

mv_train

In [ ]:
test_mask = np.logical_not(train_mask)

mv_test = sps.csr_matrix((mv_all.data[test_mask], (mv_all.row[test_mask], mv_all.col[test_mask])))

mv_test

#### Evaluation metric

#### We call items in the test set 'relevant'

In [ ]:
user_id = 124
relevant_items = mv_test[user_id].indices
relevant_items

#### Assume that we have a recommendation list such as:

In [ ]:
recommended_items = np.array([241, 1622, 15, 857, 5823])
recommended_items

In [ ]:
is_relevant = np.in1d(recommended_items, relevant_items, assume_unique=True)
is_relevant

#### Precision: how many of the recommended items are relevant

In [50]:
def precision(recommended_items, relevant_items):
    
    is_relevant = np.in1d(recommended_items,relevant_items)

    precision_score = np.sum(is_relevant) / len(is_relevant)

    return precision_score

#### Recall: how many of the relevant items I was able to recommend

In [51]:
def recall(recommended_items, relevant_items):

    is_relevant = np.in1d(recommended_items,relevant_items)

    recall_score = np.sum(is_relevant) / relevant_items.shape[0]

    return recall_score

#### Average Precision

In [52]:
def AP(recommended_items, relevant_items):

    is_relevant = np.in1d(recommended_items, relevant_items, assume_unique=True)

    # Cumulative sum: precision at 1, at 2, at 3 ...
    p_at_k = is_relevant * np.cumsum(is_relevant, dtype=np.float32) / (1 + np.arange(is_relevant.shape[0]))

    ap_score = np.sum(p_at_k) / np.min([relevant_items.shape[0], is_relevant.shape[0]])

    return ap_score

#### Put evaluate algorithm

In [ ]:
# We pass as paramether the recommender class

def evaluate_algorithm(mv_test, recommender_object, at=5):

    cumulative_precision = 0.0
    cumulative_recall = 0.0
    cumulative_AP = 0.0

    num_eval = 0


    for user_id in range(mv_test.shape[0]):

        relevant_items = mv_test.indices[mv_test.indptr[user_id]:mv_test.indptr[user_id+1]]

        if len(relevant_items)>0:

            recommended_items = recommender_object.recommend(user_id, at=at)
            num_eval += 1

            cumulative_precision += precision(recommended_items, relevant_items)
            cumulative_recall += recall(recommended_items, relevant_items)
            cumulative_AP += AP(recommended_items, relevant_items)

    cumulative_precision /= num_eval
    cumulative_recall /= num_eval
    MAP = cumulative_AP / num_eval

    print("Recommender results are: Precision = {:.4f}, Recall = {:.4f}, MAP = {:.4f}".format(
        cumulative_precision, cumulative_recall, MAP))


#### We already built a random recommender in the previous lab

#### Now, let's build a Top Popular (TopPop) recommender

### Top Popular (TopPop) recommender

#### We recommend to all users the most popular items, that is those with the highest number of interactions
#### In this case our model is the item popularity

In [ ]:
class TopPopRecommender(object):

  def fit(self, mv_train):

      item_popularity = np.ediff1d(mv_train.tocsc().indptr)

      # We are not interested in sorting the popularity value,
      # but to order the items according to it
      self.popular_items = np.argsort(item_popularity)
      self.popular_items = #FILL IN -- hint: use np.flip

  def recommend(self, user_id, at=5):

      recommended_items = #FILL IN -- hint: return top-K from self.popular_items

      return recommended_items

#### Now train and test our model

In [ ]:
topPopRecommender = TopPopRecommender()
topPopRecommender.fit(mv_train)

In [ ]:
for user_id in range(10):
    print(#FILL IN

In [ ]:
evaluate_algorithm(#FILL IN

### That's better! But we can still improve

##### <u>Hint</u>: remove items already seen by the user. We can either remove them from the recommended item list or we can set them to a score so low that it will cause them to end at the very bottom of all the available items

In [ ]:
class TopPopRecommender(object):

    def fit(self, mv_train):

        self.mv_train = mv_train

        item_popularity = #FILL IN

        # We are not interested in sorting the popularity value,
        # but to order the items according to it
        self.popular_items = #FILL IN
        self.popular_items = #FILL IN


    def recommend(self, user_id, at=5, remove_seen=True):

        if remove_seen:
            seen_items = #FILL IN -- hint: use self.mv_train.indices[x:y]

            unseen_items_mask = np.in1d(self.popular_items, seen_items, assume_unique=True, invert=True)

            unseen_items = #FILL IN -- hint: use unseen_items_mask above

            recommended_items = #FILL IN

        else:
            recommended_items = #FILL IN


        return recommended_items

In [ ]:
topPopRecommender_removeSeen = TopPopRecommender()
topPopRecommender_removeSeen.fit(mv_train)

for user_id in range(10):
    print(#FILL IN

In [ ]:
evaluate_algorithm(#FILL IN

#### Simple but effective. Always remove seen items if your purpose is to recommend "new" ones

## Section 2: Global Effect recommender

#### We recommend to all users the highest rated items

#### First we compute the average of all ratings, or global average

In [ ]:
globalAverage = np.mean(mv_train.data)

print("The global average is {:.2f}". #FILL IN

#### We substract this bias to all ratings

In [ ]:
mv_train_unbiased = #FILL IN -- hint: we don't want to override the mv_train; we need to copy to a new variable here

mv_train_unbiased.data -= #FILL IN

print(mv_train_unbiased.data[0:10])

#### Then we compute the average rating for each item, or item bias

In [ ]:
item_mean_rating = mv_train_unbiased.mean(axis=0)
item_mean_rating

In [ ]:
import matplotlib.pyplot as pyplot

item_mean_rating = np.array(item_mean_rating).squeeze()
item_mean_rating = np.sort(item_mean_rating[item_mean_rating!=0])

pyplot.plot(item_mean_rating, 'ro')
pyplot.ylabel('Item Bias')
pyplot.xlabel('Item Index')
pyplot.show()

#### And the average rating for each user, or user bias

In [ ]:
user_mean_rating = #FILL IN
user_mean_rating

In [ ]:
user_mean_rating = #FILL IN
user_mean_rating = #FILL IN

pyplot.plot(user_mean_rating, 'ro')
pyplot.ylabel('User Bias')
pyplot.xlabel('User Index')
pyplot.show()

#### Now we can sort the items by their item bias and use the same recommendation principle as in TopPop recommender

In [ ]:
class GlobalEffectsRecommender(object):

    def fit(self, mv_train):

        self.mv_train = mv_train

        globalAverage = np.mean(mv_train.data)

        mv_train_unbiased = #FILL IN
        mv_train_unbiased.data -= #FILL IN

        item_mean_rating = mv_train_unbiased.mean(axis=0)
        item_mean_rating = np.array(item_mean_rating).squeeze()

        self.bestRatedItems = np.argsort(item_mean_rating)
        self.bestRatedItems = #FILL IN -- hint: use np.flip()


    def recommend(self, user_id, at=5, remove_seen=True):

        if remove_seen:

            unseen_items_mask = np.in1d(self.bestRatedItems, mv_train[user_id].indices, assume_unique=True, invert = True)

            unseen_items = #FILL IN

            recommended_items = #FILL IN

        else:
            recommended_items = #FILL IN


        return recommended_items

In [ ]:
globalEffectsRecommender = GlobalEffectsRecommender()
globalEffectsRecommender.fit(mv_train)

evaluate_algorithm(mv_test, globalEffectsRecommender)

#### Now let's try to combine user bias and item bias

In [ ]:
class GlobalEffectsRecommender(object):

    def fit(self, mv_train):

        self.mv_train = mv_train

        globalAverage = #FILL IN

        mv_train_unbiased = #FILL IN
        mv_train_unbiased.data -= #FILL IN

        # User Bias
        user_mean_rating = mv_train_unbiased.mean(axis=1)
        user_mean_rating = np.array(user_mean_rating).squeeze()

        # In order to apply the user bias we have to change the rating value
        # in the mv_train_unbiased inner data structures
        # If we were to write:
        # mv_train_unbiased[user_id].data -= user_mean_rating[user_id]
        # we would change the value of a new matrix with no effect on the original data structure
        for user_id in range(len(user_mean_rating)):
            start_position = mv_train_unbiased.indptr[user_id]
            end_position = mv_train_unbiased.indptr[user_id+1]

            mv_train_unbiased.data[start_position:end_position] -= user_mean_rating[user_id]

        # Item Bias
        item_mean_rating = #FILL IN
        item_mean_rating = #FILL IN

        self.bestRatedItems = #FILL IN
        self.bestRatedItems = #FILL IN



    def recommend(self, user_id, at=5, remove_seen=True):

        if remove_seen:

            unseen_items_mask = #FILL IN

            unseen_items = #FILL IN

            recommended_items = #FILL IN

        else:
            recommended_items = #FILL IN


        return recommended_items


In [ ]:
globalEffectsRecommender = GlobalEffectsRecommender()
globalEffectsRecommender.fit(mv_train)

evaluate_algorithm(#FILL IN

#### The results are indeed identical! User bias is essential in case of rating prediction but not really relevant in case of top-K recommendations.

### Question:

#### Why is GlobalEffect performing worse than TopPop even if we are taking into account more information about the interaction?

### Answer:

#### The test data contains a lot of low rating interactions... We are testing against those as well, but GlobalEffects is penalizing interactions with low rating

#### In reality we want to recommend items rated in a positive way, so let's build a new Test set with positive interactions only

In [ ]:
mv_test_positiveOnly = #FILL IN -- hint: copy from mv_test

mv_test_positiveOnly.data[mv_test.data<=2] = 0
mv_test_positiveOnly.eliminate_zeros()
mv_test_positiveOnly

In [ ]:
print("Deleted {} negative interactions". #FILL IN

#### Run the evaluation again for both

In [ ]:
evaluate_algorithm(#FILL IN

In [ ]:
evaluate_algorithm(#FILL IN

### Sometimes ratings are not really more informative than interactions, depends on their quality

## Section 3: Item-based Collaborative Filtering (with Surprise)

#### Dataset: https://grouplens.org/datasets/movielens/100k/

In [ ]:
!pip install scikit-surprise

In [ ]:
import numpy as np
import pandas as pd
import heapq
from surprise import Dataset, Reader
from surprise import KNNBasic
from surprise.similarities import cosine, msd, pearson
from surprise import accuracy
from surprise.model_selection import train_test_split

In [ ]:
# If file exists, skip the download
data_100k_file_path = "movielens/Movielens100K/"
data_100k_file_name = #FILL IN

# If directory does not exist, create
if not os.path.exists(data_100k_file_path):
    os.makedirs(data_100k_file_path)

if not os.path.exists(data_100k_file_name):
    urlretrieve(#FILL IN

In [ ]:
# unzip and read the u.data file
dataFile = zipfile.ZipFile(data_100k_file_path + "ml-100k.zip")

u_rating_path = dataFile.extract(#FILL IN

In [ ]:
columns = ['user_id', 'item_id', 'rating', 'timestamp']
data = pd.read_csv(#FILL IN

In [ ]:
data.head()

#### Surprise has its own data input handle methods, and we need to use reader to read the dataframe and convert to Surprise trainset.

#### Since we are care doing item-based, care about similarity between items, we change the column order

In [ ]:
reader = Reader(rating_scale = (1, 5))
item_based_data = Dataset.load_from_df(data[['item_id', 'user_id', 'rating']], reader)

In [ ]:
full_trainset = item_based_data.build_full_trainset()

In [ ]:
trainSet, testSet = train_test_split(#FILL IN

trainSet

#### We will use KNNBasic (https://surprise.readthedocs.io/en/stable/knn_inspired.html) to predict the missing ratings

#### Note that since we have change the column order while we 'convert' the data (i.e., item_based_data above), we need to set 'user_based' to True here.

#### Please read the documentation of KNNBasic in Surprise for better understanding

In [ ]:
KNNBasic_fullTrainSet = KNNBasic(#FILL IN -- hint: look at the KNNBasic documentation above
KNNBasic_fullTrainSet.sim_options = {'name':'pearson', 'user_based':True}

KNNBasic_fullTrainSet.fit(#FILL IN -- hint: we want to fit the full trainset

In [ ]:
simMatrix_KNNBasic = #FILL IN -- hint: compute similarities from KNNBasic_fullTrainSet

print('KNNBasic similarity matrix shape: ', simMatrix_KNNBasic.shape)

print('\nPrint the first 5 rows and 5 columns of the KNNBasic similarity matrix:')
print(simMatrix_KNNBasic[0:4][0:4])

### Let's find similarity items

In [ ]:
# Import the user_id and movie_title mapping data
u_item_path = dataFile.extract(#FILL IN -- hint: check the README file again to see which file to load
items = pd.read_csv(#FILL IN
items = items.iloc[:, 0:2]
items.columns = ['item_id', 'movie_title']

In [ ]:
print('Print part of the item_id and the corresponding movie_title:')
print(items.iloc[0:9, :])

In [ ]:
# Select a movie and later we will recommend movies similar to this
print('\n Select a movie that I like, and find its item_id')
print(items[items['movie_title'].str.contains('Star Wars')])

In [ ]:
# Similarity of this selected movie and the others
sim_selected = #FILL IN -- hint: get the index above
print('\n Similarities based on the selected movie \n', sim_selected)

In [ ]:
# Find the indices of the k most similar items
k = 10
k_sim_idx_heap = []
for i in range(len(sim_selected)):
    if i == 49:
        continue
    if len(k_sim_idx_heap) < k:
        heapq.heappush(k_sim_idx_heap, (sim_selected[i], i))
    else:
        if sim_selected[i] > k_sim_idx_heap[0][0]:
            heapq.heappop(k_sim_idx_heap)
            heapq.heappush(k_sim_idx_heap, (sim_selected[i], i))

print(k_sim_idx_heap)

In [ ]:
# print out the corresponding movie_title
# since it is min heap, when printing we print from the last item to the first item
print('\n The recommemed movies are:')
for i in range(len(k_sim_idx_heap)-1, -1, -1):
    idx = k_sim_idx_heap[i][1]
    print(items.iloc[idx, 1])

## Secion 4: User-based Collaborative Filtering

#### In this section, we will walk through another movie rating example for user-based collaborative filtering

In [ ]:
!gdown 1lFHrK_sr9fyPr_2XmAg1g5cknH4UJc0W

data_file_name = "movie_rating.csv"

rating = pd.read_csv(data_file_name)
rating.head()

#### We will first create the matrix with titles of movies as rows and critics (i.e., users) as columns. Each cell contains the rating from the corresponding user for a rating

In [ ]:
rp = #FILL IN -- hint: use rating.pivot_table()
rp

#### The next step is to find the similarity score between the critics. We will use _Toby_ as example, and use _Pearson correlation_ score.

<u>Note</u>: Pandas contains the function `corrwith()` which compute the correlation

In [ ]:
rating_toby = rp['Toby']
sim_toby = #FILL IN -- hint: use .corrwith()
sim_toby

#### As you can see from the result above, Toby's taste is similar to Lisa Rose but not so much wit Gene Seymour.

#### To make recommendation for Toby, we calculate a rating of others weighted by the similarity. Note that we only need to calculate rating for movies Toby has not yet seen.

In [ ]:
# we first filter out irrelevant data, then assign the similarity score and the weighted rating
rating_c = rating.loc[rating_toby[rating.title].isnull().values & (rating.critic != 'Toby')]
rating_c_similarity = rating_c['critic'].map(sim_toby)

rating_c = rating_c.assign(similarity=rating_c_similarity, sim_rating=rating_c_similarity * rating_c.rating)
rating_c.head()

#### Lastly, we add up the score for each title using `groupby()`. We also normalize the score by dividing it with the sum of the weights.

#### Base on other critics' similarity and their rating, we have made a movie recommendation for Toby:

In [ ]:
recommendations = #FILL IN -- hint: use rating_c.groupby('title').apply(lambda xxxxx)
recommendations.sort_index(ascending=False)

# End of Lab!

References:

[1] Surprise: A Python scikit for recommender systems (https://surpriselib.com/)

[2] Programming Collective Intelligence. Toby Segaran 2007.

[3] RecSys Lab @ Polimi